In [ ]:
# ================================
# 1. Import Required Libraries
# ================================
import numpy as np
import re
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Flatten
from tensorflow.keras.optimizers import Adam

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab') # Added to resolve LookupError

# ================================
# 2. Sample Corpus
# ================================
corpus = [
    "I love machine learning and deep learning.",
    "Natural language processing is fun and interesting.",
    "Word embeddings help in NLP tasks like text classification."
]

# ================================
# 3. Text Preprocessing
# ================================
def preprocess_text(corpus):
    stop_words = set(stopwords.words('english'))
    cleaned_sentences = []

    for sentence in corpus:
        sentence = sentence.lower()
        sentence = re.sub(f"[{string.punctuation}]", "", sentence)
        words = word_tokenize(sentence)
        words = [word for word in words if word not in stop_words]
        cleaned_sentences.append(words)

    return cleaned_sentences

cleaned_corpus = preprocess_text(corpus)
print("Cleaned Corpus:", cleaned_corpus)

# ================================
# 4. Generate Skip-Gram Pairs
# ================================
def generate_context_pairs(cleaned_corpus, window_size=2):
    pairs = []

    for sentence in cleaned_corpus:
        for i, target_word in enumerate(sentence):
            start = max(0, i - window_size)
            end = min(len(sentence), i + window_size + 1)
            context_words = [sentence[j] for j in range(start, end) if j != i]

            for context_word in context_words:
                pairs.append((target_word, context_word))

    return pairs

pairs = generate_context_pairs(cleaned_corpus, window_size=2)

# ================================
# 5. Build Vocabulary
# ================================
vocab = sorted(set(word for sentence in cleaned_corpus for word in sentence))
word_to_idx = {word: idx for idx, word in enumerate(vocab)}
idx_to_word = {idx: word for word, idx in word_to_idx.items()}

# Convert word pairs to index pairs
input_words = [word_to_idx[pair[0]] for pair in pairs]
output_words = [word_to_idx[pair[1]] for pair in pairs]

# ================================
# 6. Model Hyperparameters
# ================================
vocab_size = len(vocab)
embedding_dim = 50
epochs = 100
batch_size = 2

# ================================
# 7. Build Skip-Gram Model
# ================================
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=1),
    Flatten(),
    Dense(vocab_size, activation='softmax')
])

model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy')

# Convert to numpy arrays
X = np.array(input_words)
y = np.array(output_words)

# ================================
# 8. Train Model
# ================================
model.fit(X, y, epochs=epochs, batch_size=batch_size, verbose=1)

# ================================
# 9. Extract Word Embeddings
# ================================
embeddings = model.layers[0].get_weights()[0]

def get_embedding(word):
    word_idx = word_to_idx.get(word)
    if word_idx is not None:
        return embeddings[word_idx]
    else:
        return None

# Example
embedding_for_word = get_embedding("machine")
print("\nEmbedding for 'machine':\n", embedding_for_word)


In [ ]:
import numpy as np
import pandas as pd

np.random.seed(23)

mu_vec1 = np.array([0,0,0])
cov_mat1 = np.array([[1,0,0],[0,1,0],[0,0,1]])
class1_sample = np.random.multivariate_normal(mu_vec1, cov_mat1, 20)

df = pd.DataFrame(class1_sample,columns=['feature1','feature2','feature3'])
df['target'] = 1

mu_vec2 = np.array([1,1,1])
cov_mat2 = np.array([[1,0,0],[0,1,0],[0,0,1]])
class2_sample = np.random.multivariate_normal(mu_vec2, cov_mat2, 20)

df1 = pd.DataFrame(class2_sample,columns=['feature1','feature2','feature3'])

df1['target'] = 0

df = pd.concat([df, df1], ignore_index=True)

df = df.sample(40)

In [ ]:
df.head()

In [ ]:
import plotly.express as px
#y_train_trf = y_train.astype(str)
fig = px.scatter_3d(df, x=df['feature1'], y=df['feature2'], z=df['feature3'],
              color=df['target'].astype('str'))
fig.update_traces(marker=dict(size=12,
                              line=dict(width=2,
                                        color='DarkSlateGrey')),
                  selector=dict(mode='markers'))

fig.show()


In [ ]:

# Step 2 - Find Covariance Matrix
covariance_matrix = np.cov([df.iloc[:,0],df.iloc[:,1],df.iloc[:,2]])
print('Covariance Matrix:\n', covariance_matrix)

In [ ]:

# Step 3 - Finding EV and EVs
eigen_values, eigen_vectors = np.linalg.eig(covariance_matrix)

In [ ]:

eigen_values

In [ ]:

np.array([1.3536065 , 0.94557084, 0.77774573])

In [ ]:

eigen_vectors

In [ ]:

%pylab inline

from matplotlib import pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d import proj3d
from matplotlib.patches import FancyArrowPatch


class Arrow3D(FancyArrowPatch):
    def __init__(self, xs, ys, zs, *args, **kwargs):
        FancyArrowPatch.__init__(self, (0,0), (0,0), *args, **kwargs)
        self._verts3d = xs, ys, zs

    def draw(self, renderer):
        xs3d, ys3d, zs3d = self._verts3d
        xs, ys, zs = proj3d.proj_transform(xs3d, ys3d, zs3d, self.axes.get_proj())
        self.set_positions((xs[0],ys[0]),(xs[1],ys[1]))
        FancyArrowPatch.draw(self, renderer)

    def do_3d_projection(self, renderer=None):
        # This method is called by matplotlib for depth sorting 3D artists.
        # It should return a scalar representing the depth of the object.
        # We project the mid-point of the arrow to screen coordinates and return its Z-component.
        xs3d, ys3d, zs3d = self._verts3d
        mid_x = (xs3d[0] + xs3d[1]) / 2
        mid_y = (ys3d[0] + ys3d[1]) / 2
        mid_z = (zs3d[0] + zs3d[1]) / 2

        _, _, z_depth = proj3d.proj_transform(mid_x, mid_y, mid_z, self.axes.get_proj())
        return z_depth

fig = plt.figure(figsize=(7,7))
ax = fig.add_subplot(111, projection='3d')

ax.plot(df['feature1'], df['feature2'], df['feature3'], 'o', markersize=8, color='blue', alpha=0.2)
ax.plot([df['feature1'].mean()], [df['feature2'].mean()], [df['feature3'].mean()], 'o', markersize=10, color='red', alpha=0.5)
for v in eigen_vectors.T:
    a = Arrow3D([df['feature1'].mean(), df['feature2'].mean(), df['feature3'].mean()], [v[0], v[1], v[2]], [0, 0, 0], mutation_scale=20, lw=3, arrowstyle="-|>")
    ax.add_artist(a)
ax.set_xlabel('x_values')
ax.set_ylabel('y_values')
ax.set_zlabel('z_values')

plt.title('Eigenvectors')

plt.show()
